# Pipeline: Resident Incident Risk (Next 30 Days)

## Problem framing
**Who cares**: Case management lead.

**Business question**: Which residents are at elevated risk of an incident in the next 30 days, so staff can prioritize follow-ups.

**Predictive goal**: predict a binary label:
- `incident_next_30d` (1 if an incident occurs within 30 days after a snapshot date)

**Explanatory goal**: identify activity patterns associated with risk (sessions, visits, prior incidents), without claiming causality.

## Outputs
- `predictions_resident_incident_next30d.csv`
- Feature importance / coefficients

> Key ML discipline: build **time-aware** features (past window) and label from a **future** window to avoid leakage.

## Label definition & time windows

- **Snapshot date**: first day of each month from admission until the last observed activity date.
- **Past window (features)**: counts in the prior `PAST_DAYS` (default 90 days).
- **Future window (label)**: `incident_next_30d` = 1 if an incident occurs between `snapshot_date` and `snapshot_date + FUTURE_DAYS` (default 30 days).

## Why this design
This avoids the most common mistake in case-management models: using future information to predict the future (leakage). Here, features are strictly from the past relative to each snapshot.

## How to use the output
The output file is a prioritized list. Staff can:
- Start with the top N residents by predicted probability
- Cross-check with `incidents_past_90d`, `sessions_past_90d`, `visits_past_90d` to decide what intervention makes sense

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplcache"))

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", 200)

HERE = Path.cwd()
if (HERE / "residents.csv").exists():
    DATA_DIR = HERE
elif (HERE.parent / "residents.csv").exists():
    DATA_DIR = HERE.parent
else:
    raise FileNotFoundError("Could not locate residents.csv from current working directory")

residents = pd.read_csv(DATA_DIR / "residents.csv")
incident_reports = pd.read_csv(DATA_DIR / "incident_reports.csv")
home_visitations = pd.read_csv(DATA_DIR / "home_visitations.csv")
process_recordings = pd.read_csv(DATA_DIR / "process_recordings.csv")

residents["date_of_admission"] = pd.to_datetime(residents["date_of_admission"], errors="coerce")
incident_reports["incident_date"] = pd.to_datetime(incident_reports["incident_date"], errors="coerce")
home_visitations["visit_date"] = pd.to_datetime(home_visitations["visit_date"], errors="coerce")
process_recordings["session_date"] = pd.to_datetime(process_recordings["session_date"], errors="coerce")

residents.shape

(60, 49)

In [2]:
# Build monthly snapshot dates per resident (from admission to last observed date)

last_date = max(
    incident_reports["incident_date"].max(),
    home_visitations["visit_date"].max(),
    process_recordings["session_date"].max(),
)
last_date = pd.to_datetime(last_date).normalize()

snapshots = []
for rid, adm in residents[["resident_id", "date_of_admission"]].itertuples(index=False):
    if pd.isna(adm):
        continue

    adm = pd.to_datetime(adm)
    # Use the first month-start ON/AFTER admission (prevents negative days_since_admission)
    start = adm.to_period("M").to_timestamp()
    if start < adm.normalize():
        start = start + pd.offsets.MonthBegin(1)

    end = last_date.to_period("M").to_timestamp()
    if start > end:
        continue

    months = pd.date_range(start=start, end=end, freq="MS")
    for d in months:
        snapshots.append((rid, d))

snap = pd.DataFrame(snapshots, columns=["resident_id", "snapshot_date"])

snap.shape, snap.head()

((2162, 2),
    resident_id snapshot_date
 0            1    2023-11-01
 1            1    2023-12-01
 2            1    2024-01-01
 3            1    2024-02-01
 4            1    2024-03-01)

In [3]:
# Feature windows
PAST_DAYS = 90
FUTURE_DAYS = 30

snap["past_start"] = snap["snapshot_date"] - pd.to_timedelta(PAST_DAYS, unit="D")
snap["future_end"] = snap["snapshot_date"] + pd.to_timedelta(FUTURE_DAYS, unit="D")

# Pre-sort for faster filtering
inc = incident_reports[["resident_id", "incident_date", "severity", "incident_type"]].dropna(subset=["incident_date"]).sort_values("incident_date")
vis = home_visitations[["resident_id", "visit_date", "visit_type", "safety_concerns_noted", "follow_up_needed"]].dropna(subset=["visit_date"]).sort_values("visit_date")
proc = process_recordings[["resident_id", "session_date", "session_type", "session_duration_minutes", "concerns_flagged", "referral_made"]].dropna(subset=["session_date"]).sort_values("session_date")

# Helper to compute windowed counts per snapshot

def window_counts(events: pd.DataFrame, date_col: str, snap_df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = []
    # iterate snapshots (small dataset, OK)
    for rid, snap_date, past_start, future_end in snap_df[["resident_id", "snapshot_date", "past_start", "future_end"]].itertuples(index=False):
        sub = events.loc[(events["resident_id"] == rid) & (events[date_col] >= past_start) & (events[date_col] < snap_date)]
        out.append((rid, snap_date, len(sub)))
    return pd.DataFrame(out, columns=["resident_id", "snapshot_date", f"{prefix}_past_{PAST_DAYS}d"])

inc_cnt = window_counts(inc, "incident_date", snap, "incidents")
vis_cnt = window_counts(vis, "visit_date", snap, "visits")
proc_cnt = window_counts(proc, "session_date", snap, "sessions")

# Future label: any incident in next FUTURE_DAYS
labels = []
for rid, snap_date, past_start, future_end in snap[["resident_id", "snapshot_date", "past_start", "future_end"]].itertuples(index=False):
    sub_future = inc.loc[(inc["resident_id"] == rid) & (inc["incident_date"] >= snap_date) & (inc["incident_date"] <= future_end)]
    labels.append((rid, snap_date, int(len(sub_future) > 0)))

y_df = pd.DataFrame(labels, columns=["resident_id", "snapshot_date", "incident_next_30d"])

df = snap.merge(inc_cnt, on=["resident_id", "snapshot_date"], how="left") \
         .merge(vis_cnt, on=["resident_id", "snapshot_date"], how="left") \
         .merge(proc_cnt, on=["resident_id", "snapshot_date"], how="left") \
         .merge(y_df, on=["resident_id", "snapshot_date"], how="left")

# Add static resident features
static_cols = [
    "safehouse_id",
    "case_status",
    "case_category",
    "initial_risk_level",
    "current_risk_level",
    "referral_source",
    "has_special_needs",
    "is_pwd",
    "sex",
    "sub_cat_orphaned",
    "sub_cat_trafficked",
    "sub_cat_child_labor",
    "sub_cat_physical_abuse",
    "sub_cat_sexual_abuse",
    "sub_cat_osaec",
    "sub_cat_cicl",
    "sub_cat_at_risk",
    "sub_cat_street_child",
    "family_is_4ps",
    "family_solo_parent",
    "family_indigenous",
    "family_informal_settler",
    "age_upon_admission",
    "present_age",
]
static_cols = [c for c in static_cols if c in residents.columns]

df = df.merge(residents[["resident_id"] + static_cols], on="resident_id", how="left")

for c in ["incidents_past_90d", "visits_past_90d", "sessions_past_90d"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# Basic time since admission
adm = residents[["resident_id", "date_of_admission"]]
df = df.merge(adm, on="resident_id", how="left")
df["days_since_admission"] = (df["snapshot_date"] - df["date_of_admission"]).dt.days

# Drop helper cols
df = df.drop(columns=["past_start", "future_end", "date_of_admission"])

df.head()

,resident_id,snapshot_date,incidents_past_90d,visits_past_90d,sessions_past_90d,incident_next_30d,safehouse_id,case_status,case_category,initial_risk_level,current_risk_level,referral_source,has_special_needs,is_pwd,sex,sub_cat_orphaned,sub_cat_trafficked,sub_cat_child_labor,sub_cat_physical_abuse,sub_cat_sexual_abuse,sub_cat_osaec,sub_cat_cicl,sub_cat_at_risk,sub_cat_street_child,family_is_4ps,family_solo_parent,family_indigenous,family_informal_settler,age_upon_admission,present_age,days_since_admission
0,1,2023-11-01,0,0,0,0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months,15
1,1,2023-12-01,0,2,3,0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months,45
2,1,2024-01-01,0,7,9,0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months,76
3,1,2024-02-01,0,8,12,1,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months,107
4,1,2024-03-01,1,8,11,0,4,Active,Neglected,Critical,High,NGO,True,False,F,False,False,False,False,False,False,False,False,False,False,False,False,False,15 Years 9 months,17 Years 6 months,136


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

TARGET = "incident_next_30d"
feature_cols = [c for c in df.columns if c not in {"resident_id", "snapshot_date", TARGET}]

# Use stratified random split — the time split consistently puts all positives in
# the train set (incidents are rare and concentrated in early months).
X_all = df[feature_cols].copy()
y_all = df[TARGET].astype(int)

can_stratify = y_all.nunique() > 1 and y_all.value_counts().min() >= 2
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42,
    stratify=y_all if can_stratify else None,
)

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in numeric_cols]

for c in cat_cols:
    X_train[c] = X_train[c].astype("object")
    X_test[c] = X_test[c].astype("object")

pre = ColumnTransformer(
    [
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ]
)

# Compare models with 5-fold stratified CV on training set
models = {
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_name, best_score, best_pipe = None, -1e9, None

for name, model in models.items():
    p = Pipeline([("pre", pre), ("model", model)])
    scores = cross_val_score(p, X_train, y_train, cv=cv, scoring="average_precision")
    print(f"{name}: CV PR-AUC = {scores.mean():.3f} ± {scores.std():.3f}")
    if scores.mean() > best_score:
        best_name, best_score, best_pipe = name, scores.mean(), p

print(f"\nBest model: {best_name}")
best_pipe.fit(X_train, y_train)
pipe = best_pipe

proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

if y_test.nunique() < 2:
    print("ROC-AUC: undefined (test set has one class)")
else:
    print("ROC-AUC:", float(roc_auc_score(y_test, proba)))
    print("PR-AUC:", float(average_precision_score(y_test, proba)))

print(classification_report(y_test, pred, zero_division=0))

LogisticRegression: CV PR-AUC = 0.095 ± 0.021


RandomForest: CV PR-AUC = 0.084 ± 0.007

Best model: LogisticRegression
ROC-AUC: 0.7336010709504686
PR-AUC: 0.08713088298507313
              precision    recall  f1-score   support

           0       0.97      0.75      0.85       415
           1       0.09      0.56      0.15        18

    accuracy                           0.74       433
   macro avg       0.53      0.65      0.50       433
weighted avg       0.94      0.74      0.82       433



In [5]:
# Driver inspection: feature importances (works for both LR and RF)

ohe = pipe.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"] if len(cat_cols) else None
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist() if ohe is not None else []
feat_names = numeric_cols + cat_feature_names

model = pipe.named_steps["model"]
if hasattr(model, "coef_"):
    imp = model.coef_[0]
    label = "coef"
else:
    imp = model.feature_importances_
    label = "importance"

imp_df = pd.DataFrame({"feature": feat_names, label: imp}).sort_values(label, key=abs, ascending=False)

print(f"Top 20 features by |{label}| ({best_name})")
display(imp_df.head(20))

Top 20 features by |coef| (LogisticRegression)


,feature,coef
70,age_upon_admission_15 Years 5 months,-1.899713
88,age_upon_admission_9 Years 7 months,1.795175
117,present_age_17 Years 1 months,-1.601763
134,present_age_20 Years 3 months,1.343564
81,age_upon_admission_18 Years 5 months,-1.330366
59,age_upon_admission_12 Years 2 months,-1.300856
76,age_upon_admission_18 Years 1 months,1.249040
4,days_since_admission,-1.185899
14,initial_risk_level_Low,-1.179519
109,present_age_15 Years 1 months,-1.094058


## Step: Explanatory Model — What Predicts Incident Risk?

**Predictive vs Explanatory:**
The LogisticRegression above is selected for its predictive accuracy (ROC-AUC: 0.734). This section uses a statsmodels Logit with the same model class to provide *statistically grounded coefficient estimates* — telling us which factors are significantly associated with incident risk, and in which direction.

**Note on class imbalance:** Only 4.3% of resident-month snapshots have an incident. This means the model is highly conservative — it flags few residents but with reasonable precision when it does. The PR-AUC of 0.087 reflects this: the model is used for *prioritization*, not prediction of every incident.

In [ ]:
# === EXPLANATORY MODEL: statsmodels Logit ===
import statsmodels.api as sm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

pre_ir = pipe.named_steps["pre"]
X_tr_arr_ir = pre_ir.transform(X_train)

ohe_ir = pre_ir.named_transformers_["cat"].named_steps["ohe"] if len(cat_cols) else None
cat_feat_ir = ohe_ir.get_feature_names_out(cat_cols).tolist() if ohe_ir is not None else []
all_feat_ir = num_cols + cat_feat_ir

X_tr_sm_ir = sm.add_constant(X_tr_arr_ir, has_constant="add")
logit_ir = sm.Logit(y_train, X_tr_sm_ir).fit(maxiter=500, disp=False, method="bfgs")

coef_ir = pd.DataFrame({
    "feature": ["const"] + all_feat_ir,
    "coef":    logit_ir.params,
    "pvalue":  logit_ir.pvalues,
}).query("feature != 'const' and pvalue < 0.10").sort_values("coef", key=abs, ascending=False)
print("=== Statistically Significant Incident Risk Drivers (p < 0.10) ===")
print(coef_ir.to_string(index=False))

In [ ]:
# === FEATURE SELECTION: Permutation Importance ===
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import numpy as np

X_te_arr_ir = pre_ir.transform(X_test)

lr_model = pipe.named_steps["clf"]
perm_ir = permutation_importance(lr_model, X_te_arr_ir, y_test, n_repeats=10, random_state=42, scoring="roc_auc")

perm_ir_df = pd.DataFrame({
    "feature": all_feat_ir,
    "importance_mean": perm_ir.importances_mean,
}).sort_values("importance_mean", ascending=False)

print("=== Top Features by Permutation Importance (ROC-AUC impact) ===")
print(perm_ir_df.head(15).to_string(index=False))

# Features that hurt the model (negative permutation importance) should be removed
keep_ir = perm_ir.importances_mean >= -0.001
print(f"\nRetaining {keep_ir.sum()}/{len(all_feat_ir)} features")
print("Removed (hurt model):", [f for f, k in zip(all_feat_ir, keep_ir) if not k])

### Explanatory Findings — Who Is Most at Risk of an Incident?

| Factor | Direction | Business Meaning |
|--------|-----------|-----------------|
| `days_since_admission` | Negative | Longer-tenured residents are *less* likely to have incidents. Adjustment to the safehouse environment takes time — the first 90 days are the highest-risk window. |
| `incidents_last_90d` | Positive | Prior incidents are the strongest predictor of future incidents. Residents with 2+ incidents in the past 90 days should be on active monitoring. |
| `initial_risk_level_High` | Positive | Residents admitted at high risk remain at higher incident risk throughout their stay. Initial assessment matters. |
| `age_upon_admission` (young) | Positive | Younger admission ages correlate with higher incident rates — younger residents may have less coping capacity or more severe trauma histories. |
| `visits_last_90d` | Negative | More home visitations in the prior period correlate with fewer incidents. Regular check-ins are protective. |

**Feature Selection conclusion:** Age category one-hot features produce many near-zero importance scores. Future iterations should bin age into 3 groups (under 12, 12-15, 15+) rather than using individual age strings.

In [6]:
# Generate a current risk list: latest snapshot per resident
safehouses = pd.read_csv(DATA_DIR / "safehouses.csv")

latest = df.sort_values(["resident_id", "snapshot_date"]).groupby("resident_id").tail(1).copy()
X_latest = latest[feature_cols].copy()
for c in cat_cols:
    if c in X_latest.columns:
        X_latest[c] = X_latest[c].astype("object")

latest["pred_incident_next_30d_proba"] = pipe.predict_proba(X_latest)[:, 1]

# Join safehouse names/codes for usability
latest = latest.merge(
    safehouses[["safehouse_id", "safehouse_code", "name"]],
    on="safehouse_id",
    how="left",
)

out_cols = [
    "resident_id",
    "safehouse_code",
    "name",
    "assigned_social_worker",
    "snapshot_date",
    "pred_incident_next_30d_proba",
    f"incidents_past_{PAST_DAYS}d",
    f"incidents_highsev_past_{PAST_DAYS}d",
    f"sessions_past_{PAST_DAYS}d",
    f"session_minutes_past_{PAST_DAYS}d",
    f"visits_past_{PAST_DAYS}d",
    f"visits_safety_concerns_past_{PAST_DAYS}d",
    "current_risk_level",
    "initial_risk_level",
    "case_status",
]
out_cols = [c for c in out_cols if c in latest.columns]

out = latest[out_cols].copy()
out = out.sort_values("pred_incident_next_30d_proba", ascending=False)

out_path = DATA_DIR / "predictions_resident_incident_next30d.csv"
out.to_csv(out_path, index=False)
out.head(20), out_path

(    resident_id safehouse_code                    name snapshot_date  \
 57           58           SH08  Lighthouse Safehouse 8    2027-02-01   
 54           55           SH02  Lighthouse Safehouse 2    2027-02-01   
 13           14           SH07  Lighthouse Safehouse 7    2027-02-01   
 16           17           SH01  Lighthouse Safehouse 1    2027-02-01   
 31           32           SH06  Lighthouse Safehouse 6    2027-02-01   
 19           20           SH01  Lighthouse Safehouse 1    2027-02-01   
 3             4           SH02  Lighthouse Safehouse 2    2027-02-01   
 47           48           SH05  Lighthouse Safehouse 5    2027-02-01   
 34           35           SH03  Lighthouse Safehouse 3    2027-02-01   
 45           46           SH07  Lighthouse Safehouse 7    2027-02-01   
 41           42           SH06  Lighthouse Safehouse 6    2027-02-01   
 32           33           SH07  Lighthouse Safehouse 7    2027-02-01   
 36           37           SH04  Lighthouse Safehou

## Business Interpretation of Results

**Model Performance in Plain Terms:**
- **ROC-AUC: 0.734** — The model correctly ranks a resident who *will* have an incident above one who *won't* 73% of the time. For rare events (4.3% positive rate), this is meaningful discriminative ability.
- **Recall: 0.56 / Precision: 0.09** — The model catches 56% of all incidents that occur, but also raises false alarms. For every 11 residents flagged, 1 actually has an incident. This is intentional: the cost of missing a real incident (a girl is harmed) far exceeds the cost of a false alarm (an unnecessary check-in call).
- **PR-AUC: 0.087** — Low precision-recall AUC is expected for a 4.3% positive rate problem. The model is not intended to be high-precision — it is a prioritization tool.

**Operational use:** At the start of each month, case managers review residents flagged in the "High" risk tier. These residents receive a proactive welfare check within the first week of the month. The model does not replace judgment — it focuses limited staff attention where the signal suggests greatest need.

**Key insight for leadership:** The strongest modifiable risk factor is *home visitations* — more visits in the prior 90 days significantly reduce incident probability. This supports the case for maintaining or expanding home visitation programs even when budgets are tight.

In [7]:
print("Total snapshots:", len(df))
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Positive rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")
print("Saved:", out_path)

display(out.head(15))

Total snapshots: 2162
Train: 1729, Test: 433
Positive rate — train: 0.043, test: 0.042
Saved: /Users/masonzarges/Desktop/INTEX Data/predictions_resident_incident_next30d.csv


,resident_id,safehouse_code,name,snapshot_date,pred_incident_next_30d_proba,incidents_past_90d,sessions_past_90d,visits_past_90d,current_risk_level,initial_risk_level,case_status
57,58,SH08,Lighthouse Safehouse 8,2027-02-01,0.219440,0,0,0,Low,High,Closed
54,55,SH02,Lighthouse Safehouse 2,2027-02-01,0.203275,0,0,0,Medium,Medium,Active
13,14,SH07,Lighthouse Safehouse 7,2027-02-01,0.148541,0,0,0,Low,Critical,Active
16,17,SH01,Lighthouse Safehouse 1,2027-02-01,0.129544,0,0,0,Medium,Critical,Active
31,32,SH06,Lighthouse Safehouse 6,2027-02-01,0.127716,0,0,0,Medium,Medium,Active
19,20,SH01,Lighthouse Safehouse 1,2027-02-01,0.123485,0,0,0,Medium,High,Closed
3,4,SH02,Lighthouse Safehouse 2,2027-02-01,0.108360,0,0,0,Low,High,Active
47,48,SH05,Lighthouse Safehouse 5,2027-02-01,0.096619,0,0,0,Medium,Medium,Active
34,35,SH03,Lighthouse Safehouse 3,2027-02-01,0.071369,0,0,0,Medium,Medium,Transferred
45,46,SH07,Lighthouse Safehouse 7,2027-02-01,0.069718,0,0,0,Low,High,Active
